# Stellar Classification (SDSS) — GPU Stacking Ensemble

A three-model stack — **LightGBM + XGBoost + TabPFN** — combined by a
logistic-regression meta-learner, for the S6E6 task (GALAXY / QSO / STAR).

TabPFN, a tabular foundation model, supplies the model diversity that a homogeneous
gradient-boosting blend lacks; a GPU runtime gives it a substantial in-context
training set. Physics-motivated color-index features feed all three base models, and
out-of-fold predictions are stacked rather than naively averaged.

*Runtime: GPU accelerator + internet. Output: `submission.csv`.*


### 1. Install TabPFN (LightGBM / XGBoost / sklearn are preinstalled on Kaggle)

In [ ]:
!pip install -q tabpfn


### 2. Imports and config

In [ ]:
import os
os.environ["TABPFN_NO_BROWSER"] = "1"
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

import numpy as np, pandas as pd, torch
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_predict
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
from xgboost import XGBClassifier

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    _cap = torch.cuda.get_device_capability()
    print("GPU:", torch.cuda.get_device_name(0), "| compute capability", _cap)
    if _cap[0] < 7:   # e.g. Tesla P100 (6.0) is too old for recent PyTorch builds
        print("WARNING: this GPU is too old for the installed PyTorch (needs >= 7.0).")
        print("TabPFN will fail with a CUDA 'no kernel image' error.")
        print("Switch the accelerator to 'GPU T4 x2' and re-run.")

CLASSES = ["GALAXY", "QSO", "STAR"]
C2I = {c: i for i, c in enumerate(CLASSES)}
I2C = {i: c for c, i in C2I.items()}
CAT = ["spectral_type", "galaxy_population"]

# auto-detect the data folder (Kaggle mounts competition data at varying paths)
import glob
_hits = glob.glob("/kaggle/input/**/train.csv", recursive=True)
DATA = os.path.dirname(_hits[0]) if _hits else "/kaggle/input/playground-series-s6e6"
print("data folder:", DATA)


### 3. Load data + color-index feature engineering (27 features)

In [ ]:
def add_features(df):
    df = df.copy(); bands = ["u", "g", "r", "i", "z"]
    df["u_g"]=df.u-df.g; df["g_r"]=df.g-df.r; df["r_i"]=df.r-df.i; df["i_z"]=df.i-df.z
    df["u_r"]=df.u-df.r; df["u_i"]=df.u-df.i; df["u_z"]=df.u-df.z
    df["g_i"]=df.g-df.i; df["g_z"]=df.g-df.z; df["r_z"]=df.r-df.z
    df["mag_mean"]=df[bands].mean(1); df["mag_std"]=df[bands].std(1)
    df["mag_range"]=df[bands].max(1)-df[bands].min(1)
    df["z_x_ug"]=df.redshift*df.u_g; df["z_x_gr"]=df.redshift*df.g_r; df["z_x_ri"]=df.redshift*df.r_i
    df["redshift_log"]=np.log1p(df.redshift.clip(lower=0))
    return df

train = add_features(pd.read_csv(f"{DATA}/train.csv"))
test  = add_features(pd.read_csv(f"{DATA}/test.csv"))
y = train["class"].map(C2I).values

for c in CAT:
    train[c] = train[c].astype("category")
    test[c]  = pd.Categorical(test[c], categories=train[c].cat.categories)

feats = [c for c in train.columns if c not in ["id", "class"]]
Xg, Xtg = train[feats], test[feats]                       # GBM view (native categoricals)

both = pd.get_dummies(pd.concat([train[feats], test[feats]]), columns=CAT, dummy_na=True)
Xl  = both.iloc[:len(train)].to_numpy("float32")          # numeric view for TabPFN
Xtl = both.iloc[len(train):].to_numpy("float32")
print(len(feats), "features |", len(train), "train rows")


### 4. Base models — LightGBM & XGBoost (5-fold OOF)

In [ ]:
# Put the tree models on the GPU when one is present (XGBoost is reliable;
# LightGBM's GPU build may be absent, so probe it and fall back to CPU).
def _lgb_device():
    if DEVICE != "cuda":
        return "cpu"
    try:
        lgb.LGBMClassifier(device="gpu", n_estimators=1, verbose=-1).fit(
            np.random.rand(80, 4), np.random.randint(0, 2, 80))
        return "gpu"
    except Exception as e:
        print("LightGBM GPU unavailable, using CPU:", str(e)[:60])
        return "cpu"

LGB_DEVICE = _lgb_device()
XGB_DEVICE = "cuda" if DEVICE == "cuda" else "cpu"
print("tree devices -> LightGBM:", LGB_DEVICE, "| XGBoost:", XGB_DEVICE)

LGB = dict(objective="multiclass", num_class=3, learning_rate=0.03, num_leaves=127,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=2.0,
           reg_alpha=0.5, min_child_samples=40, n_estimators=3000,
           device=LGB_DEVICE, random_state=42, n_jobs=-1, verbose=-1)
XGB = dict(objective="multi:softprob", num_class=3, learning_rate=0.05, max_depth=8,
           subsample=0.8, colsample_bytree=0.7, reg_lambda=2.0, reg_alpha=0.5,
           n_estimators=3000, tree_method="hist", device=XGB_DEVICE,
           enable_categorical=True, eval_metric="mlogloss",
           early_stopping_rounds=80, random_state=42, n_jobs=-1)

skf = StratifiedKFold(5, shuffle=True, random_state=42)

def oof(fit_predict, X, Xt):
    o = np.zeros((len(y), 3)); t = np.zeros((len(Xt), 3))
    for tr, va in skf.split(np.zeros(len(y)), y):
        pv, pt = fit_predict(tr, va, X, Xt); o[va] = pv; t += pt / 5
    return o, t

def fp_lgb(tr, va, X, Xt):
    m = lgb.LGBMClassifier(**LGB)
    m.fit(X.iloc[tr], y[tr], eval_set=[(X.iloc[va], y[va])],
          callbacks=[lgb.early_stopping(80, verbose=False)])
    return m.predict_proba(X.iloc[va]), m.predict_proba(Xt)

def fp_xgb(tr, va, X, Xt):
    m = XGBClassifier(**XGB)
    m.fit(X.iloc[tr], y[tr], eval_set=[(X.iloc[va], y[va])], verbose=False)
    return m.predict_proba(X.iloc[va]), m.predict_proba(Xt)

oof_lgb, te_lgb = oof(fp_lgb, Xg, Xtg); print("LightGBM OOF:", accuracy_score(y, oof_lgb.argmax(1)))
oof_xgb, te_xgb = oof(fp_xgb, Xg, Xtg); print("XGBoost  OOF:", accuracy_score(y, oof_xgb.argmax(1)))


### 5. TabPFN base model (GPU)

The foundation-model base learner. `CTX` sets the in-context training size,
trading accuracy against GPU memory.

In [ ]:
from huggingface_hub import hf_hub_download
from tabpfn import TabPFNClassifier

ckpt = hf_hub_download(repo_id="Prior-Labs/tabpfn_3",
                       filename="tabpfn-v3-classifier-v3_default.ckpt")

CTX = 16000 if DEVICE == "cuda" else 5000   # TabPFN training-context size
NEST = 4 if DEVICE == "cuda" else 1
PREC = "auto"   # TabPFN auto-selects mixed precision on GPU (safe on any device)

def fp_tab(tr, va, X, Xt):
    sub, _ = train_test_split(tr, train_size=min(CTX, len(tr)),
                              random_state=0, stratify=y[tr])
    clf = TabPFNClassifier(model_path=ckpt, device=DEVICE, inference_precision=PREC,
                           ignore_pretraining_limits=True, n_estimators=NEST)
    clf.fit(X[sub], y[sub])
    return clf.predict_proba(X[va]), clf.predict_proba(Xt)

oof_tab, te_tab = oof(fp_tab, Xl, Xtl)
print("TabPFN OOF:", accuracy_score(y, oof_tab.argmax(1)))


### 6. Stack with a logistic-regression meta-learner, write submission

In [ ]:
meta_X  = np.hstack([oof_lgb, oof_xgb, oof_tab])
meta_Xt = np.hstack([te_lgb,  te_xgb,  te_tab])

meta = LogisticRegression(max_iter=2000, n_jobs=-1)
stack_oof = cross_val_predict(meta, meta_X, y, cv=skf, method="predict_proba")
print("STACK CV        :", accuracy_score(y, stack_oof.argmax(1)))
print("LightGBM alone  :", accuracy_score(y, oof_lgb.argmax(1)))
print("(CV is optimistic on this competition ~0.011 - the leaderboard is the judge)")

meta.fit(meta_X, y)
pred = meta.predict(meta_Xt)
sub = pd.DataFrame({"id": pd.read_csv(f"{DATA}/test.csv")["id"],
                    "class": [I2C[i] for i in pred]})
sub.to_csv("submission.csv", index=False)
print(sub["class"].value_counts())
print("saved submission.csv")


### Notes

Writes `submission.csv`. The stack can be extended by raising `CTX` / `NEST` or adding
diverse base models (CatBoost, an MLP, a second GBM seed). Method, diagnostics, and the
CPU-vs-GPU trade-off are documented in the project's `docs/experiments.md`.
